# Fresh grounded v2 adapter — Colab training
Run on a fresh Colab T4 (or better) GPU session. This notebook starts from `meta-llama/Llama-3.2-3B-Instruct` and creates a NEW LoRA adapter; it never loads v1. Use notebook 10 locally first.

Pilot defaults: 3 epochs maximum, learning rate 5e-5, rank 16 / alpha 32, all-linear targets, effective batch 8, sequence limit 1024. These are starting settings, not proven optimal. Compare checkpoint behaviour, not just validation loss. The 24-case diagnostic benchmark is not used for training or checkpoint selection here.

Keep the notebook source in your VS Code project; execute this training notebook in Colab because the local 4 GB GPU is reserved for inference.


In [ ]:
%pip install -q -U transformers datasets accelerate peft trl bitsandbytes
%pip uninstall -y torchao


If installation asks for a runtime restart, restart before proceeding. Do not rerun installation in a session where the model is already loaded. Package versions are recorded below; this notebook has not been GPU-executed by its author.


In [ ]:
from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, importlib.metadata as metadata
import torch
from huggingface_hub import notebook_login

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
assert torch.cuda.get_device_properties(0).total_memory >= 12 * 2**30, "Use a T4 or larger GPU for this pilot."
print(torch.cuda.get_device_name(0))
drive.mount("/content/drive")
notebook_login()

# For a new experiment leave this as None. To resume, paste the SAME v2 run
# directory previously printed, and set RESUME_FROM in the training cell.
EXISTING_RUN_DIR = None
RUN_DIR = Path(EXISTING_RUN_DIR) if EXISTING_RUN_DIR else (
    Path("/content/drive/MyDrive/Agent1/grounding_v2") /
    datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
)
RUN_DIR.mkdir(parents=True, exist_ok=bool(EXISTING_RUN_DIR))
print("Run directory:", RUN_DIR)

required = ["train_v2.jsonl", "validation_v2.jsonl", "data_manifest.json", "response_prompt_snapshot.py"]
if not all((RUN_DIR / name).is_file() for name in required):
    uploaded = files.upload()
    for name in required:
        if name not in uploaded:
            raise FileNotFoundError(f"Upload {name} from notebook 10's output folder.")
        shutil.copy2(name, RUN_DIR / name)

versions = {p: metadata.version(p) for p in ["torch", "transformers", "datasets", "accelerate", "peft", "trl", "bitsandbytes"]}
(RUN_DIR / ("versions_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + ".json")).write_text(json.dumps(versions, indent=2))
print(versions)


In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

set_seed(42)
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
MAX_LENGTH = 1024
manifest = json.loads((RUN_DIR / "data_manifest.json").read_text())
raw = {}
for split in ["train", "validation"]:
    name = f"{split}_v2.jsonl"
    assert hashlib.sha256((RUN_DIR / name).read_bytes()).hexdigest() == manifest["files"][name]
    raw[split] = [json.loads(s) for s in (RUN_DIR / name).read_text().splitlines() if s.strip()]
assert len(raw["train"]) == 240 and len(raw["validation"]) == 48
assert not ({r["group_id"] for r in raw["train"]} & {r["group_id"] for r in raw["validation"]})

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Fail rather than silently truncating or dropping training targets.
for split, rows in raw.items():
    lengths = [len(tokenizer.apply_chat_template(r["prompt"] + r["completion"], tokenize=True, add_generation_prompt=False)) for r in rows]
    print(split, "rows:", len(rows), "maximum tokens:", max(lengths))
    assert max(lengths) <= MAX_LENGTH, "An example exceeds MAX_LENGTH; review before training."

# Only prompt and completion reach SFT; IDs, groups, and classes remain metadata.
dataset = DatasetDict({s: Dataset.from_list([
    {"prompt": r["prompt"], "completion": r["completion"]} for r in rows
]) for s, rows in raw.items()})
dataset["train"] = dataset["train"].shuffle(seed=42)


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quantization,
    device_map={"": 0}, dtype=compute_dtype,
    attn_implementation="eager",
)
base_model = prepare_model_for_kbit_training(base_model)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", bias="none", task_type="CAUSAL_LM",
)
# Fresh adapter initialization. No PeftModel.from_pretrained(v1) call.
model = get_peft_model(base_model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()
tokenizer.save_pretrained(str(RUN_DIR / "tokenizer"))
(RUN_DIR / "model_setup.json").write_text(json.dumps({
    "base_model": MODEL_ID, "base_revision": getattr(model.config, "_commit_hash", None),
    "fresh_adapter": True, "rank": 16, "alpha": 32,
    "targets": "all-linear", "quantization": quantization.to_dict(),
}, indent=2))


## Compatibility and validation previews
The old T4 run required disabling an optimized TRL matrix multiplication. Apply that workaround only if the installed TRL still exposes the flag and the GPU is older than compute capability 8.0. This is a private compatibility flag, not a training hyperparameter.

Eight fixed validation examples (one per class) are generated after each evaluation and saved to Drive. They are diagnostic previews, not automatic semantic scores; inspect all 48 validation cases when choosing a final candidate.


In [ ]:
import importlib
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

if torch.cuda.get_device_capability(0)[0] < 8:
    try:
        chunked_logprob = importlib.import_module("trl.kernels.chunked_logprob")
    except ModuleNotFoundError:
        chunked_logprob = None
    if chunked_logprob is not None and hasattr(chunked_logprob, "_MM_OUT_DTYPE"):
        chunked_logprob._MM_OUT_DTYPE = False
        print("Applied the previously used T4 compatibility workaround.")

preview_rows = [next(r for r in raw["validation"] if r["prompt_type"] == kind)
                for kind in sorted({r["prompt_type"] for r in raw["validation"]})]

class SaveValidationPreviews(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        was_training = model.training
        model.eval()
        previews = []
        try:
            for row in preview_rows:
                inputs = tokenizer.apply_chat_template(
                    row["prompt"], tokenize=True, add_generation_prompt=True,
                    return_dict=True, return_tensors="pt",
                ).to("cuda")
                with torch.inference_mode():
                    ids = model.generate(**inputs, max_new_tokens=128, do_sample=False,
                                         use_cache=True, pad_token_id=tokenizer.pad_token_id)
                new_ids = ids[0, inputs["input_ids"].shape[1]:].tolist()
                eos = model.generation_config.eos_token_id
                if eos is None: eos = tokenizer.eos_token_id
                eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos])
                previews.append({
                    "id": row["id"], "prompt_type": row["prompt_type"],
                    "prompt": row["prompt"], "target": row["completion"][0]["content"],
                    "answer": tokenizer.decode(new_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False),
                    "hit_token_limit": len(new_ids) >= 128 and (not new_ids or new_ids[-1] not in eos_ids),
                })
            path = RUN_DIR / f"validation_preview_step_{state.global_step:04}.json"
            path.write_text(json.dumps(previews, ensure_ascii=False, indent=2))
            print("Saved", path.name)
        finally:
            model.train(was_training)
        return control

training_args = SFTConfig(
    output_dir=str(RUN_DIR / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    bf16=use_bf16,
    fp16=not use_bf16,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=30,
    save_strategy="steps",
    save_steps=15,
    save_total_limit=6,
    load_best_model_at_end=False,
    report_to="none",
    seed=42,
)
trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=dataset["train"], eval_dataset=dataset["validation"],
    processing_class=tokenizer, callbacks=[SaveValidationPreviews()],
)
(RUN_DIR / "training_config.json").write_text(training_args.to_json_string())


In [ ]:
# Inspect which tokens actually receive training loss BEFORE training.
example = trainer.train_dataset[0]
assert "completion_mask" in example, "Expected a prompt/completion mask; inspect installed TRL behaviour."
features = {k: example[k] for k in ["input_ids", "attention_mask", "completion_mask", "assistant_masks"] if k in example}
batch = trainer.data_collator([features])
labels = batch["labels"][0]
input_ids = batch["input_ids"][0]
mask = torch.tensor(example["completion_mask"], dtype=torch.bool)
assert (labels[:len(mask)][~mask] == -100).all(), "Prompt tokens must not receive loss."
assert (labels != -100).any(), "No supervised completion tokens!"
print("SUPERVISED TOKENS (should be only the assistant answer and its end marker):")
print(tokenizer.decode(input_ids[labels != -100], skip_special_tokens=False))
print("Supervised token count:", int((labels != -100).sum()))


Stop if the supervised text contains the system instruction, user question, or an empty answer. Otherwise run the next cell. With 240 rows, batch 1 and accumulation 8, there are 30 optimizer updates per epoch and 90 total. Checkpoints are saved every 15 updates; validation loss and response previews every 30. The checkpoint saving interval limits lost work after a disconnect.


In [ ]:
# Leave None for fresh training. To resume an interrupted v2 run, set
# EXISTING_RUN_DIR above and supply one of THAT run's checkpoint directories.
RESUME_FROM = None
if RESUME_FROM is not None:
    resume_path = Path(RESUME_FROM)
    assert resume_path.parent.resolve() == (RUN_DIR / "checkpoints").resolve()
    assert (resume_path / "trainer_state.json").is_file()
    assert (resume_path / "optimizer.pt").is_file(), "Need a full trainer checkpoint to resume."
else:
    assert not list((RUN_DIR / "checkpoints").glob("checkpoint-*")), "Use resume mode or a new run directory."
    baseline_metrics = trainer.evaluate()
    (RUN_DIR / "baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2))

train_result = trainer.train(resume_from_checkpoint=RESUME_FROM)
# The final step (90) already triggers validation and preview saving.
(RUN_DIR / "training_metrics.json").write_text(json.dumps(train_result.metrics, indent=2))
(RUN_DIR / "trainer_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))

# This is the LAST adapter, not an automatically approved best model.
adapter_dir = RUN_DIR / "adapter_last"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved last adapter:", adapter_dir)
print("Checkpoint candidates:", sorted(p.name for p in (RUN_DIR / "checkpoints").glob("checkpoint-*")))


## What to inspect next
Download `trainer_log.json` and `validation_preview_step_0000.json`, `0030.json`, `0060.json`, `0090.json` (actual filenames include `validation_preview_step_`). Compare context use, useful clarification, unsupported claims, and verbosity across epochs. Inspect full validation responses before selecting a candidate; lower loss alone is not enough.

Keep the original v1 adapter unchanged. Copy the selected v2 adapter/tokenizer folder into `models/grounding_v2/adapter` for local inference, after review. Evaluate with the same prompt snapshot and generation settings as notebook 09. GPU/dtype/library differences between Colab and local inference can affect outputs, so do the final base-versus-v2 comparison locally. An independent final test remains future work.
